<a href="https://colab.research.google.com/github/DipaliKhochare/Generative_AI_Applications_with_RAG_and_LangChain/blob/main/Generative_AI_Applications_with_RAG_and_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# installing necessary packages compatible with Colab's Python 3.13 environment
# Note: Dependency resolver conflict warnings for google-colab/google-adk can be safely ignored.
!pip install \
gradio \
ibm-watsonx-ai \
langchain \
langchain-community \
langchain-ibm \
chromadb \
pypdf \
pydantic \
huggingface_hub

***Import the necessary libraries***

In [2]:
# Import the required libraries into the notebook environment
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from langchain_ibm import WatsonxEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_23856/1463142952.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
%%writefile qabot.py

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from ibm_watsonx_ai import Credentials
from langchain_ibm import WatsonxLLM, WatsonxEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from huggingface_hub import HfFolder

import gradio as gr

# Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

Overwriting qabot.py


***Initialize the LLM***

In [4]:
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from langchain_ibm import WatsonxLLM
from google.colab import userdata

def get_llm():
    model_id = 'ibm/granite-4-h-small'
    parameters = {
        GenParams.MAX_NEW_TOKENS: 256,   # max tokens the LLM can generate in a response
        GenParams.TEMPERATURE: 0.5,      # controls randomness (lower = more deterministic)
    }
    project_id = "skills-network"

    # Securely retrieve the key from Colab's secret manager
    api_key = userdata.get('WATSONX_API_KEY')

    watsonx_llm = WatsonxLLM(
        model_id=model_id,
        url="https://us-south.ml.cloud.ibm.com",
        apikey=api_key,
        project_id=project_id,
        params=parameters,
    )
    return watsonx_llm

***Document Loader using Langchain (PyPDFLoader)***

In [5]:
## This QA bot is reagrding Indian labour law Act
## Recently indian government converted indian labour Act into 4 codes. All these codes are attached below in the form pdf.
## which are -
## 1.The Code on Wages 2019
## 2.The Industrial Relations Code 2020
## 3.The Code on Social Security 2020
## 4.The Occupational Safety, Health and Working Conditions (OSH&WC) Code 2020
## Document loader
def document_loader(labour_law):
  urls = [
    "https://www.labour.gov.in/static/uploads/2025/06/c328da14bbb15fc4ad571dc33e7a4ab3.pdf",
    "https://www.labour.gov.in/static/uploads/2025/07/682a44b5426bff2c1f4943ee1b2fd566.pdf",
    "https://www.labour.gov.in/static/uploads/2025/07/b0620548445580767b5c0d18c95c26f7.pdf",
    "https://www.labour.gov.in/static/uploads/2025/07/e0971b414d2951d58dd958ecec9c8c09.pdf"
  ]

  loaded_documents = []
  for url in urls:
    loader = PyPDFLoader(url)
    loaded_documents.extend(loader.load())

  return loaded_documents

***Text Splitter: Apply text splitting techniques***

In [6]:
## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

***Embed documents: Define the embedding model***

In [7]:
## Embedding model
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from google.colab import userdata

def watsonx_embedding():
    embed_params = {
        EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS_TO_MAX_LIMIT: True
    }

    # Securely retrieve the key from Colab's secret manager
    api_key = userdata.get('WATSONX_API_KEY')

    watsonx_embedding = WatsonxEmbeddings(
        model_id="ibm/slate-125m-english-rtrvr",
        url="https://us-south.ml.cloud.ibm.com",
        apikey=api_key,
        project_id="skills-network",
        params=embed_params,
    )
    return watsonx_embedding

***Create and configure vector databases to store embeddings***

In [8]:
## Vector db
def vector_database(chunks):
    embedding_model = watsonx_embedding()
    vectordb = Chroma.from_documents(documents=chunks, embedding=embedding_model)
    return vectordb

***Define the retriever: Develop a retriever to fetch document segments based on queries***

In [9]:
## Retriever
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever

***Define a question-answering chain: QA Bot that leverages the LangChain and LLM to answer questions***

In [10]:
## QA Chain
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)
    qa = RetrievalQA.from_chain_type(llm=llm,
                                    chain_type="stuff",
                                    retriever=retriever_obj,
                                    return_source_documents=True)
    response = qa.invoke({"query": query})
    return response['result'], response.get('source_documents', [])

***Gradio User Interface***
Build a friendly user interface to interact with our Indian Labour Codes chatbot.

In [11]:
import gradio as gr

# Close any stale active Gradio connections to release bound ports
gr.close_all()

def chat_interface(labour_law_file, query):
    if not query.strip():
        return "Please enter a valid question.", ""
    try:
        # Unpack both the main answer and the retrieved source document chunks
        response, source_docs = retriever_qa(labour_law_file, query)

        # Format source documents nicely for display
        formatted_sources = ""
        for i, doc in enumerate(source_docs, 1):
            metadata = doc.metadata
            page = metadata.get('page', 'N/A')
            source = metadata.get('source', 'N/A')
            # Grab first 200 characters of each snippet to keep UI clean
            snippet = doc.page_content[:200].strip().replace('\n', ' ') + "..."
            formatted_sources += f"Source {i} (Page {page}):\n{snippet}\n\n"

        return response, formatted_sources if formatted_sources else "No source documents retrieved."
    except Exception as e:
        return f"An error occurred: {str(e)}", ""

# Defining the Gradio interface with two outputs
rag_application = gr.Interface(
    fn=chat_interface,
    inputs=[
        gr.Dropdown(
            choices=["The Code on Wages 2019", "The Industrial Relations Code 2020", "The Code on Social Security 2020", "The Occupational Safety, Health and Working Conditions Code 2020"],
            label="Select Labour Code",
            value="The Code on Wages 2019"
        ),
        gr.Textbox(label="Ask a question about Indian Labour Codes", placeholder="e.g., What are the rules regarding minimum wages?")
    ],
    outputs=[
        gr.Textbox(label="Chatbot Answer"),
        gr.Textbox(label="Retrieved Source Documents", placeholder="Sources will be displayed here...")
    ],
    title="Indian Labour Codes QA Assistant",
    description="Ask questions directly based on the newly converted Indian Labour Code PDF documents via Watsonx LLM."
)

# Launch the interface, letting Gradio find an empty port dynamically if 7860 is taken
rag_application.launch(share=True, server_name="0.0.0.0")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be1f290f23538cf4e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
